# Week 2 · Day 2 — Cleaning & transforming data

*Real data is messy. Make it usable.*

**By the end you'll have shipped:** a **cleaned coffee-orders table** — text normalized, `"$4.50"` turned into real numbers, gaps filled, duplicates removed — saved and ready to analyze.

> Core Path = everything unmarked. `Go Deeper 🔧` = optional.
> Builds on **Day 1 (pandas fundamentals)** — you can't `groupby` a column of `"$4.50"` strings until you clean it.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1 · Python foundations (Week 2) |
| **Prerequisites** | Week 2 Day 1 — pandas fundamentals |
| **Est. time** | ~30 min |
| **Capstone tie-in** | *Matter Intelligence* — real exports arrive messy; cleaning is step one of every pipeline |
| **Difficulty** | Core (+ optional Go Deeper) |

### 🎯 Learning objectives

By the end you'll be able to:
- **Spot** data-quality problems with `.info()`, `.isna()`, and a look at the values.
- **Fix types** — turn `"$4.50"` text into real numbers with `to_numeric`.
- Handle **missing data** with `isna` / `fillna` / `dropna`.
- **Normalize text** with the `.str` accessor (`strip`, `title`).
- Remove **duplicates** and add **derived columns**.

### ⚖️ Why it matters

The tidy `coffee_orders.csv` from Day 1 was a gift. Real exports aren't like that. A point-of-sale system spits out `latte` and `LATTE` and `  Mocha `; prices arrive as text like `"$4.50"`; some cells are just **empty**; the same order gets logged **twice**.

Try to `groupby("item")` on that and you'll get *three* different "lattes" and a crash when you sum a column of strings. **Cleaning is the unglamorous step that makes every later step possible** — and it's most of the job in real data work. Today we take a deliberately messy file and make it analysis-ready.

### ⚙️ Setup

Loads the **messy** `coffee_orders_raw.csv` (shared copy if available, else a built-in messy sample). We keep the original around as `raw` so you can compare before/after.

In [ ]:
import os
import pandas as pd

# A deliberately messy built-in sample — used only if the shared raw CSV isn't found.
MESSY = [
    {"order_id":"O-6001","date":"2026-03-02","item":"latte","size":"M","category":"Espresso Drink","price":"$4.75","payment":"Card","store":"Downtown"},
    {"order_id":"O-6002","date":"2026-03-02","item":"LATTE","size":"L","category":"Espresso Drink","price":"5.50","payment":"app","store":"Uptown"},
    {"order_id":"O-6003","date":"2026-03-03","item":"  Mocha ","size":"S","category":"Espresso Drink","price":"$4.50","payment":"Cash","store":"Downtown"},
    {"order_id":"O-6005","date":"2026-03-04","item":"Drip","size":"","category":"Brewed","price":"2.95","payment":"Card","store":"Uptown"},
    {"order_id":"O-6006","date":"2026-03-04","item":"Cappuccino","size":"L","category":"Espresso Drink","price":"","payment":"Cash","store":"Downtown"},
    {"order_id":"O-6008","date":"2026-03-05","item":"Latte ","size":"S","category":"Espresso Drink","price":"4.00","payment":"Card","store":"Downtown"},
    {"order_id":"O-6008","date":"2026-03-05","item":"Latte ","size":"S","category":"Espresso Drink","price":"4.00","payment":"Card","store":"Downtown"},
    {"order_id":"O-6009","date":"2026-03-06","item":"ESPRESSO","size":"M","category":"Espresso Drink","price":"$3.25","payment":"app","store":"uptown"},
    {"order_id":"O-6012","date":"2026-03-07","item":"mocha","size":"L","category":"Espresso Drink","price":"$5.95","payment":"Card","store":""},
]

SHARED = os.path.join("..", "..", "data", "coffee_orders_raw.csv")
if os.path.exists(SHARED):
    raw = pd.read_csv(SHARED, dtype=str, keep_default_na=False, na_values=[""])
else:
    raw = pd.DataFrame(MESSY).replace("", pd.NA)

print("pandas", pd.__version__, "— loaded", len(raw), "raw rows")
raw

### 1 · Look before you leap — spot the problems

Never clean blind. `.info()` shows types and how many non-null values each column has; `.isna().sum()` counts the gaps. Read the table above and you'll already see three issues: **inconsistent item text**, **prices stored as strings**, and **empty cells**.

In [ ]:
raw.info()
print("\nMissing values per column:")
print(raw.isna().sum())

**What just happened:** every column came in as `object` (text) — including `price`, which should be a number. `isna().sum()` flags the empty `price` and `store` cells. Now we fix them one at a time. We'll work on a copy so the original `raw` stays intact for comparison.

In [ ]:
df = raw.copy()

### 2 · Fix the type — `"$4.50"` → a real number  (SQL: `CAST`)

You can't `sum` or `groupby`-average a column of strings. Strip the `$` (and any commas), then `pd.to_numeric` converts to float. `errors="coerce"` turns anything unparseable into `NaN` instead of crashing.

In [ ]:
df["price"] = (
    df["price"].astype(str)
               .str.replace("$", "", regex=False)
               .str.replace(",", "", regex=False)
               .str.strip()
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")   # text -> float, bad values -> NaN

print("price dtype now:", df["price"].dtype)
print(df["price"].head())

### 3 · Handle missing values  →  `isna` / `fillna` / `dropna`  (SQL: `COALESCE`)

Now `price` has a `NaN` (the empty Cappuccino cell). You have three moves: **detect** (`isna`), **fill** (`fillna`), or **drop** (`dropna`). There's no universal right answer — it depends on the data. Here we fill the missing price with the column **median** (a reasonable stand-in) and fill the missing store label with `"Unknown"`.

In [ ]:
print("rows with a missing price:")
print(df[df["price"].isna()][["order_id", "item", "price"]])

median_price = df["price"].median()
df["price"] = df["price"].fillna(round(median_price, 2))
df["store"] = df["store"].fillna("Unknown")

print("\nmissing after fill:", df["price"].isna().sum(), "prices,", df["store"].isna().sum(), "stores")

> **`Go Deeper 🔧` — when to drop instead.** If a row is missing something you can't responsibly guess (a missing `order_id`, say), drop it: `df.dropna(subset=["order_id"])`. Filling invents data; dropping loses it. Choose deliberately and write down which you did.

### 4 · Normalize text  →  the `.str` accessor  (SQL: `TRIM`, `INITCAP`)

`latte`, `LATTE`, and `  Mocha ` should all be one clean value. The `.str` accessor runs string methods down the whole column at once: `.str.strip()` removes stray spaces, `.str.title()` gives consistent capitalization.

In [ ]:
print("before:", sorted(df["item"].unique()))

df["item"] = df["item"].str.strip().str.title()
df["store"] = df["store"].str.strip().str.title()
df["payment"] = df["payment"].str.strip().str.title()

print("after: ", sorted(df["item"].unique()))

**What just happened:** the three spellings of "latte" collapsed into one `"Latte"`. *This* is why cleaning comes before grouping — now `groupby("item")` will count lattes correctly instead of splitting them three ways.

### 5 · Remove duplicates  →  `drop_duplicates`  (SQL: `DISTINCT`)

Order `O-6008` was logged twice. `duplicated()` flags repeats; `drop_duplicates()` removes them.

In [ ]:
print("duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("rows after de-dup:", len(df))

### 6 · Add a derived column  →  `assign`

Cleaning often ends with **enrichment** — computing a new field the raw data didn't have. `assign` returns a new frame with the column added (tidy and chainable). Here: a simple `price_tier` label.

In [ ]:
df = df.assign(
    price_tier = df["price"].apply(lambda x: "premium" if x >= 5 else "standard")
)
df[["order_id", "item", "price", "price_tier"]].head()

> **`Common pitfalls ⚠️`**
>
> - `to_numeric` on dirty text **errors** unless you strip symbols first or pass `errors="coerce"`.
> - `fillna` vs `dropna` is a real decision — filling *invents* data. Be intentional and note it.
> - `.str` methods on a column with `NaN` return `NaN` for those cells (they don't crash) — clean or fill first if that matters.
> - `drop_duplicates()` keeps the **first** copy by default; use `subset=[...]` to dedupe on specific columns (e.g. `order_id`).

### ✍️ Your turn

In [ ]:
# Start fresh from the messy data:
work = raw.copy()

# TODO 1: clean the 'price' column into real numbers (strip '$', then pd.to_numeric)
# TODO 2: normalize 'item' text (strip + title case) and print the unique values
# TODO 3: fill any missing 'store' with "Unknown"
# TODO 4 (stretch): after cleaning, what is the average price per item? (groupby + mean)

# your code here


<details><summary>✅ Show solution</summary>

```python
work["price"] = pd.to_numeric(
    work["price"].astype(str).str.replace("$", "", regex=False).str.strip(),
    errors="coerce",
)
work["item"] = work["item"].str.strip().str.title()
print(sorted(work["item"].unique()))
work["store"] = work["store"].fillna("Unknown")
print(work.groupby("item")["price"].mean().round(2))
```
</details>

### 🚀 Build the artifact — a cleaned, analysis-ready table

Chain the whole cleaning pipeline and save the result. This `coffee_orders_clean.csv` is what Day 3 will group and join — clean input is the foundation of every report.

In [ ]:
clean = raw.copy()

# types
clean["price"] = pd.to_numeric(
    clean["price"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip(),
    errors="coerce",
)
# missing
clean["price"] = clean["price"].fillna(round(clean["price"].median(), 2))
clean["store"] = clean["store"].fillna("Unknown")
# text
for col in ["item", "store", "payment"]:
    clean[col] = clean[col].str.strip().str.title()
# duplicates
clean = clean.drop_duplicates().reset_index(drop=True)

print(f"raw rows: {len(raw)}  ->  clean rows: {len(clean)}")
print("price dtype:", clean["price"].dtype, "| items:", sorted(clean["item"].unique()))

clean.to_csv("coffee_orders_clean.csv", index=False)
print("\n✅ Shipped: coffee_orders_clean.csv")

> **🔗 Your world — from coffee to matters.** A raw matters export has the exact same warts: `amount_billed` as `"$18,500.00"` text, `Contracts`/`contracts`/`CONTRACTS`, blank `lead_attorney` cells, a matter entered twice. The moves are identical — `to_numeric` the billed amount, `.str.title()` the practice area, `fillna` the gaps, `drop_duplicates` the repeats. Clean once, and every report downstream is trustworthy.

### 📝 Recap — what you shipped

- **Inspect first:** `.info()` + `.isna().sum()` reveal type and gap problems.
- **Fix types:** strip symbols, then `pd.to_numeric(..., errors="coerce")`.
- **Missing data:** `fillna` (invent a stand-in) vs `dropna` (lose the row) — a real choice.
- **Normalize text:** `.str.strip().str.title()` collapses spelling variants.
- **De-dup & enrich:** `drop_duplicates()` and `assign` for new columns.
- **Artifact:** a saved `coffee_orders_clean.csv`.

### 🧠 Check your understanding

1. Why must you clean the `price` column *before* you can `groupby(...).sum()` it?
2. What does `errors="coerce"` do in `pd.to_numeric`?
3. What's the trade-off between `fillna` and `dropna`?

<details><summary>Answers</summary>

1. It arrives as **text** (`"$4.50"`); you can't sum or average strings. `to_numeric` turns it into a real number first.
2. It turns any value that **can't be parsed** into `NaN` instead of raising an error — so one bad cell doesn't crash the whole conversion.
3. `fillna` keeps the row but **invents** a value (bias risk); `dropna` keeps only real data but **loses** rows (and any signal in them). Pick based on the column and document it.
</details>

### ➡️ Next up — Week 2, Day 3: grouping, aggregation & joins

With a clean table you can finally do the powerful stuff: multi-level `groupby`, `pivot_table`, and **joining** the orders to a `menu` table to compute **profit margin**. Then Day 4 rebuilds it all in **Polars**.

### 📖 Reference & glossary

| Term | Plain meaning | SQL twin |
|---|---|---|
| `pd.to_numeric` | text → number | `CAST(x AS NUMERIC)` |
| `isna` / `fillna` | find / fill gaps | `IS NULL` / `COALESCE` |
| `dropna` | drop rows with gaps | `WHERE x IS NOT NULL` |
| `.str.strip` / `.str.title` | trim / re-case text | `TRIM` / `INITCAP` |
| `drop_duplicates` | remove repeat rows | `DISTINCT` |
| `assign` | add a computed column | `SELECT x, ... AS new` |

**Official docs:** [Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) · [`to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) · [Working with text](https://pandas.pydata.org/docs/user_guide/text.html)